# Spark 101 — Spark Connect on Jetson via k3s

## How this setup works

This notebook runs PySpark against a **Spark Connect** server deployed on **k3s** (lightweight Kubernetes), all on a single Jetson Orin Nano 8GB. Three components are orchestrated together:

### 1. Spark Server (k3s pod)
The Spark 4.1.2 Connect server runs inside a Kubernetes pod managed by k3s. It listens on port `15002` (gRPC) and runs in `local[*]` mode — all compute happens in-process inside the pod. No separate executor pods.

```
k3s cluster (single-node)
└── spark namespace
    └── spark-connect Deployment
        └── Spark Connect Server (JVM)
            ├── gRPC endpoint :15002
            └── Spark UI :4040
```

**Manifests**: `experiments/spark-k8s/k8s/` — the entire deployment is declarative YAML, one `kubectl apply` to reproduce.

### 2. k3s (Kubernetes)
Installed on the Jetson with `curl -sfL https://get.k3s.io | sh -`. Provides:
- **Container lifecycle** — auto-restarts the Spark pod on crash
- **Health checks** — readiness/liveness probes on port 15002
- **Config management** — Spark properties live in a ConfigMap, editable without rebuilding images
- **Future scaling** — when adding nodes, change one line (`spark.master = k8s://...`) and Spark schedules executor pods across the cluster

### 3. PySpark client (.venv)
The notebook's virtualenv has the `pyspark` pip package installed as a **thin client only**. It connects to the Spark server over gRPC — no local JVM, no heavy Spark installation on the host.

```python
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
```

### How they connect

```
JupyterLab (.venv)                   k3s cluster
┌──────────────────┐                ┌─────────────────────────┐
│  PySpark client   │── gRPC ──────▶│  Spark Connect Server   │
│  (no JVM)         │   :15002      │  (JVM, local[*] mode)   │
└──────────────────┘                └─────────────────────────┘
                                     Spark UI at :4040
```

### Key decisions made during setup

| Decision | Why |
|----------|-----|
| `local[*]` instead of `k8s://` master | Single-node — executor pods are pure overhead with no distributed benefit |
| k3s over plain Docker | Declarative manifests, health checks, and zero-friction path to multi-node |
| Spark Connect (gRPC) over embedded Spark | Notebook stays lightweight — no JVM in the Python process |
| `spark.driver.bindAddress = 0.0.0.0` | Required in K8s — the pod can't bind to the Service ClusterIP, only to its own interfaces |

## Comparison: This Setup vs Databricks Free Edition (Serverless)

| | **This Setup (Jetson + k3s)** | **Databricks Community / Free (Serverless)** |
|---|---|---|
| **Where compute runs** | On your hardware — Spark JVM in a k3s pod on the Jetson | Databricks-managed cloud VMs (AWS/Azure/GCP) spun up on demand |
| **Architecture** | Spark Connect server + thin gRPC client | Serverless SQL warehouse or classic cluster — Databricks manages everything |
| **Startup time** | Pod already running (~0s) | Cold start: 30s–2min while VMs provision |
| **Cost** | Electricity only (~5W idle) | Free tier has limited DBUs; paid tiers bill per DBU-hour |
| **Catalog / Storage** | None by default — in-memory DataFrames, read from local files | Unity Catalog, Delta Lake, managed cloud storage (S3/ADLS/GCS) |
| **Scaling** | Manual — add k3s nodes, change one config line | Automatic — serverless scales executors transparently |
| **Data persistence** | Ephemeral unless you write to disk/configure a metastore | Delta tables persist in cloud storage across sessions |
| **GPU access** | Direct — Jetson's GPU is right there for CUDA/PyTorch | GPU clusters available but expensive; not on free tier |
| **Network latency** | Zero — everything is local | Depends on region; data transfer costs apply |
| **Control** | Full — you own the JVM flags, the container, the OS | Databricks controls the runtime; you configure through their UI/API |

### What Databricks Serverless actually does differently

Databricks "serverless" isn't just Spark — it's a **managed, opinionated platform** on top of Spark:

1. **Pre-warmed pools** — VMs are provisioned before you need them, so "serverless" means fast cold starts, not no servers
2. **Unity Catalog** — centralized governance, access control, and lineage tracking across all data assets
3. **Delta Lake** — ACID transactions, time travel, schema enforcement on top of Parquet — this is what makes `spark.read.table()` work seamlessly
4. **Photon engine** — Databricks' native C++ execution engine replaces parts of Spark's JVM execution for faster queries
5. **Notebooks + MLflow** — integrated experiment tracking, model registry, and deployment

### What this Jetson setup gives you that Databricks doesn't

- **No cloud dependency** — runs air-gapped, on your desk, on Tailscale, anywhere
- **GPU-integrated workflows** — Spark for data prep → CUDA/PyTorch for compute, same machine, no data transfer
- **Full infrastructure learning** — you understand every layer: k3s, pods, Spark configs, gRPC, networking
- **Unlimited experimentation** — no DBU limits, no session timeouts, no cluster auto-termination

### Bottom line

Databricks is the right choice when you need **managed data infrastructure at scale** — catalogs, governance, Delta Lake, team collaboration, and you're okay paying for it.

This Jetson setup is the right choice when you want **full control, local GPU access, and zero ongoing cost** — and you're willing to manage the infrastructure yourself. It's also a better learning environment because nothing is abstracted away.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

# Create some sample data
transactions = spark.createDataFrame([
    (1, "txn001", 150.00),
    (1, "txn002", 200.00),
    (2, "txn003", 75.50),
    (2, "txn004", 300.00),
    (2, "txn005", 50.00),
    (3, "txn006", 425.00),
], schema="account_id INT, txn_id STRING, amount DOUBLE")

transaction_summary = (transactions.groupBy("account_id").agg(count("txn_id").alias("txn_count"),sum("amount").alias("total_amount")))

transaction_summary.show()

+----------+---------+------------+
|account_id|txn_count|total_amount|
+----------+---------+------------+
|         1|        2|       350.0|
|         2|        3|       425.5|
|         3|        1|       425.0|
+----------+---------+------------+

